In [ ]:
# BLIP 모델

In [1]:
pip install transformers pillow torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
import re
import pandas as pd
from PIL import Image
import requests
from transformers import BlipProcessor, BlipForConditionalGeneration
from tqdm import tqdm

In [3]:
# 1. 모델 로드(이미지 캡셔닝)
model_name = "Salesforce/blip-image-captioning-large"
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

In [ ]:
# 이미지 불러오기 (웹에서)
#img_url = "https://contents.kyobobook.co.kr/sih/fit-in/458x0/pdt/9791199305304.jpg"
#image = Image.open(requests.get(img_url, stream=True).raw)

In [8]:
# 2. 엑셀 불러오기

from google.colab import files
uploaded = files.upload()

Saving Blip test50.xlsx to Blip test50.xlsx


In [12]:
df = pd.read_excel("Blip test50.xlsx")
results = []

In [13]:
# (추가) 프롬프트 설정 (자세한 설명 요청)
prompt = "a detailed description of the image:"

In [14]:
# 3. 각 이미지 처리
for idx, row in tqdm(df.iterrows(), total=len(df)):
    img_url = row['이미지 URL']  # 컬럼명 정확히 맞추기
    try:
        image = Image.open(requests.get(img_url, stream=True).raw)

        # 캡션 생성
        inputs = processor(images=image, return_tensors="pt")
        out = model.generate(**inputs, max_length=50)
        caption = processor.decode(out[0], skip_special_tokens=True)

        # 키워드 추출 (간단 후처리)
        words = re.findall(r"\b[a-zA-Z]+\b", caption.lower())
        stopwords = {"a", "an", "the", "of", "on", "and", "image", "with", "in", "under", "front", "back", "behind", "it", "keywords", "list", "describing", "next", "detailed", "description", "ly","poster", "movie","s", "for", "background","up", "close", "picture", "painting", "to", "go"}
        keywords = [w for w in words if w not in stopwords]

         # 결과 저장
        results.append({"이미지 URL": img_url, "caption": caption, "keywords": ", ".join(keywords)})
    except Exception as e:
        results.append({"이미지 URL": img_url, "caption": "Error", "keywords": "Error"})

100%|██████████| 48/48 [12:09<00:00, 15.20s/it]


In [15]:
# 4. 원본 데이터프레임에 결과 추가
result_df = pd.concat([df, pd.DataFrame(results)], axis=1)

In [16]:
# 5. 엑셀 저장
result_df.to_excel("book_image_analysis.xlsx", index=False)
print("✅ 완료! → book_image_analysis.xlsx 저장됨")

✅ 완료! → book_image_analysis.xlsx 저장됨


In [17]:
from google.colab import files
files.download("book_image_analysis.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>